# Week 6: ML Pipelines and Hyperparameter Tuning

## Objectives
1. Build complete ML pipelines with ColumnTransformer
2. Perform hyperparameter tuning with GridSearchCV
3. Analyze permutation importance
4. Serialize models for deployment

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, classification_report
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("✅ Imports complete!")

## 2. Load Data

In [ ]:
# Load dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Preprocess
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop('customerID', axis=1)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"Dataset: {df.shape}")
df.head()

## 3. Feature Types

In [ ]:
# Identify feature types
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in df.columns 
                       if col not in numeric_features + ['Churn']]

print(f"Numeric ({len(numeric_features)}): {numeric_features}")
print(f"Categorical ({len(categorical_features)}): {categorical_features}")

## 4. Build Pipeline with ColumnTransformer

In [ ]:
# Create ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
         categorical_features)
    ])

# Create full pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Pipeline created!")
print(pipeline)

## 5. Train/Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 6. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5]
}

print("Parameter Grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

print(f"\nTotal combinations: {np.prod([len(v) for v in param_grid.values()])}")

In [ ]:
# Setup GridSearchCV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("Fitting GridSearchCV...")
grid_search.fit(X_train, y_train)
print("\n✅ Grid search complete!")

In [ ]:
# Best results
print(f"Best CV Score (F1): {grid_search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

## 7. Evaluate Best Model

In [ ]:
# Get best model
best_model = grid_search.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Metrics
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_prob)
}

print("Test Set Metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Classification report
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

## 8. Grid Search Results Analysis

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

# Display top 10 configurations
cols = ['param_classifier__n_estimators', 'param_classifier__max_depth', 
        'param_classifier__min_samples_split', 'mean_test_score', 'std_test_score']
print("Top 10 Configurations:")
print(results_df[cols].sort_values('mean_test_score', ascending=False).head(10).to_string())

In [ ]:
# Visualize hyperparameter effects
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Effect of n_estimators
pivot1 = results_df.pivot_table(
    values='mean_test_score', 
    index='param_classifier__n_estimators',
    aggfunc='mean'
)
pivot1.plot(kind='bar', ax=axes[0], legend=False)
axes[0].set_title('Effect of n_estimators')
axes[0].set_ylabel('F1 Score')

# Effect of max_depth
pivot2 = results_df.pivot_table(
    values='mean_test_score',
    index='param_classifier__max_depth',
    aggfunc='mean'
)
pivot2.plot(kind='bar', ax=axes[1], legend=False)
axes[1].set_title('Effect of max_depth')
axes[1].set_ylabel('F1 Score')

# Effect of min_samples_split
pivot3 = results_df.pivot_table(
    values='mean_test_score',
    index='param_classifier__min_samples_split',
    aggfunc='mean'
)
pivot3.plot(kind='bar', ax=axes[2], legend=False)
axes[2].set_title('Effect of min_samples_split')
axes[2].set_ylabel('F1 Score')

plt.tight_layout()
plt.show()

## 9. Permutation Importance

In [ ]:
# Calculate permutation importance
print("Calculating permutation importance...")

result = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring='f1'
)

# Get feature names
try:
    feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
except:
    # Fallback
    feature_names = numeric_features + list(
        best_model.named_steps['preprocessor']
        .named_transformers_['cat']
        .get_feature_names_out(categorical_features)
    )

# Create importance DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values('importance_mean', ascending=False)

print("\nTop 15 Features:")
print(importance_df.head(15).to_string(index=False))

In [ ]:
# Plot permutation importance
plt.figure(figsize=(10, 8))
top_features = importance_df.head(15)

plt.barh(range(len(top_features)), top_features['importance_mean'], 
         xerr=top_features['importance_std'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Permutation Importance')
plt.title('Top 15 Feature Importances (Permutation)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/permutation_importance_plot.png', dpi=150)
plt.show()

## 10. Model Serialization

In [ ]:
# Save the best pipeline
import os
os.makedirs('../outputs', exist_ok=True)

pipeline_path = '../outputs/best_pipeline.pkl'
joblib.dump(best_model, pipeline_path)
print(f"✅ Pipeline saved to {pipeline_path}")

# Also save grid search results
results_df.to_csv('../outputs/grid_search_results.csv', index=False)
importance_df.to_csv('../outputs/permutation_importance.csv', index=False)
print("✅ Results saved!")

In [ ]:
# Demonstrate loading and using the saved model
loaded_model = joblib.load(pipeline_path)

# Make prediction on new data
sample = X_test.iloc[:5]
predictions = loaded_model.predict(sample)
probabilities = loaded_model.predict_proba(sample)[:, 1]

print("Sample Predictions:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    print(f"  Sample {i+1}: Churn={pred} (probability={prob:.3f})")

## 11. Compare Multiple Models

In [ ]:
# Compare different classifiers
classifiers = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

model_results = []

for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    
    # Create pipeline
    pipe = Pipeline([
        ('preprocessor', ColumnTransformer([
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
             categorical_features)
        ])),
        ('classifier', clf)
    ])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Evaluate
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    model_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(model_results)
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(comparison_df.round(4).to_string(index=False))

In [ ]:
# Visualize comparison
comparison_df.set_index('Model').plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. Summary

### What We Learned
1. **Pipelines** ensure consistent preprocessing between train and test
2. **ColumnTransformer** handles different feature types elegantly
3. **GridSearchCV** automates hyperparameter tuning
4. **Permutation Importance** provides reliable feature importance
5. **joblib** enables easy model serialization

### Best Practices
- Always use pipelines to prevent data leakage
- Use `handle_unknown='ignore'` for production robustness
- Cross-validation gives better performance estimates
- Save feature names alongside the model